# SVM — The Library Version

Same linear model via scikit-learn's `SVC`, verifying the scratch build — then the industrial kernel: RBF on the circle data our hand-lift solved, plus the γ knob's failure modes (README §4).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

df = pd.read_csv("data/qc_data.csv")
X = df[["temperature_c", "vibration_mm_s"]].values
y = np.where(df.label.values == "fail", 1, -1)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=55)
sc = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)

lin = SVC(kernel="linear", C=100).fit(Xtr_s, ytr)     # C ~ 1/lambda: 100 ~ our lam=0.01 regime
print(f"linear SVC test accuracy: {accuracy_score(yte, lin.predict(Xte_s)):.0%}  <- matches the scratch build")
print(f"support vectors: {lin.n_support_.sum()} of {len(ytr)}   (the health meter, README §5)")
print(f"learned w = {lin.coef_[0].round(2)}, b = {lin.intercept_[0]:.2f}")

linear SVC test accuracy: 98%  <- matches the scratch build
support vectors: 18 of 160   (the health meter, README §5)
learned w = [1.87 4.26], b = -0.27


In [2]:
# The real kernel trick: RBF on circle-in-a-ring (our Block 9 data), no hand-lifting required.
rs = np.random.default_rng(8)
r_in = rs.uniform(0, 1.1, 120); a_in = rs.uniform(0, 2*np.pi, 120)
r_out = rs.uniform(1.7, 2.6, 120); a_out = rs.uniform(0, 2*np.pi, 120)
Xc = np.vstack([np.c_[r_in*np.cos(a_in), r_in*np.sin(a_in)],
                np.c_[r_out*np.cos(a_out), r_out*np.sin(a_out)]])
yc = np.array([1]*120 + [-1]*120)

for kernel, note in [("linear", "no street exists in 2D"),
                     ("rbf", "the implicit infinite lift — no manual x1^2+x2^2 needed")]:
    m = SVC(kernel=kernel, C=1).fit(Xc, yc)
    print(f"{kernel:6s}: {accuracy_score(yc, m.predict(Xc)):.0%}   <- {note}")

print()
# gamma's failure modes (README §4): similarity reach
for g, note in [(0.001, "tiny gamma: everyone 'similar' -> over-smooth (underfit)"),
                (1, "sensible"),
                (500, "huge gamma: islands around single points — the SVM's K=1 (overfit)")]:
    m = SVC(kernel="rbf", C=1, gamma=g).fit(Xc, yc)
    sv_frac = m.n_support_.sum() / len(yc)
    print(f"gamma={g:>6}: train acc {accuracy_score(yc, m.predict(Xc)):.0%}, SVs {sv_frac:.0%}   <- {note}")

linear: 70%   <- no street exists in 2D
rbf   : 100%   <- the implicit infinite lift — no manual x1^2+x2^2 needed

gamma= 0.001: train acc 72%, SVs 100%   <- tiny gamma: everyone 'similar' -> over-smooth (underfit)
gamma=     1: train acc 100%, SVs 10%   <- sensible
gamma=   500: train acc 100%, SVs 99%   <- huge gamma: islands around single points — the SVM's K=1 (overfit)


**The takeaway:** `SVC(kernel='linear')` is Blocks 4–6 of the scratch notebook solved by a specialized optimizer; `kernel='rbf'` is Block 9's lift, industrialized to infinite dimensions via pairwise similarities. C and γ are the two dials — width-vs-violations and reach-of-similarity — and the support-vector count remains your free health meter.